In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
import builtins

In [0]:
dbutils.widgets.text("catalogo", "catalog_project_ssm")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")
dbutils.widgets.text("file_info_operacional", "renovacion_prestamo_clientes_info_operacional")
dbutils.widgets.text("file_score_financiero", "renovacion_prestamo_clientes_score_financiero")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")
file_info_operacional = dbutils.widgets.get("file_info_operacional")
file_score_financiero = dbutils.widgets.get("file_score_financiero")

In [0]:
df_info_operacional = spark.table(f"{catalogo}.{esquema_source}.{file_info_operacional}")
df_score_financiero = spark.table(f"{catalogo}.{esquema_source}.{file_score_financiero}")

In [0]:
df_joined = df_info_operacional.join(
    df_score_financiero, 
    on="ID", 
    how="left"
)

In [0]:
df_renovacion = df_joined.withColumn("CLIENTE", F.col("ID"))
df_renovacion = df_renovacion.drop("ID")

In [0]:
COLUMNAS_IMPUTAR_NEGATIVOS = ['Ahorro','Prestamo_vigente','Promed_6Mdeuda']

COLUMNAS_TRANSFORMAR_LOG = ['Uso_Linea',
                            'Uso_TrimLinea',
                            'Saldo_Consumo',
                            'SUELDO_ESTIMADO',
                            'ANTIGUEDAD_MES',
                            'Linea_Renovado',
                            'Ahorro',
                            'Prestamo_vigente',
                            'Promed_6Mdeuda',
                            'SUELDO_ESTIMADO',
                            'Deuda_Cubierta']

COLUMNAS_NULOS_SAMPLING_ALEATORIO = ['Uso_TrimLinea','Uso_Linea','Uso_TrimLinea_LOG','Uso_Linea_LOG','Meses_oferta']

COLUMNAS_NULOS_MEDIANA=['Saldo_Consumo',
                        'SUELDO_ESTIMADO',
                        'ANTIGUEDAD_MES',
                        'Saldo_Consumo_LOG',
                        'SUELDO_ESTIMADO_LOG',
                        'ANTIGUEDAD_MES_LOG',
                        'EDAD',
                        'Prestamo_vigente_LOG'
                        ]

COLUMNAS_CATEGORIAS = ['REGION','SEXO','EST_CIVIL']

COLUMNAS_KMEANS = ['Uso_TrimLinea_LOG', 'Prestamo_vigente_LOG', 'Uso_Linea_LOG']

COLUMNAS_DROP=['Uso_Linea',
               'Uso_TrimLinea',
               'Saldo_Consumo',
               'SUELDO_ESTIMADO',
               'ANTIGUEDAD_MES',
               'Linea_Renovado',
               'Ahorro',
               'Prestamo_vigente',
               'Promed_6Mdeuda',
               'Deuda_Cubierta',
               'MES',
               'CLIENTE']

In [0]:
def imputar_columnas_negativas(df: DataFrame) -> DataFrame:
    for col in COLUMNAS_IMPUTAR_NEGATIVOS:
        df = df.withColumn(col, F.greatest(F.lit(0), F.col(col)))
    return df

In [0]:
def transformar_columnas(df: DataFrame) -> DataFrame:
    for col in COLUMNAS_TRANSFORMAR_LOG:
        new_col_name = f'{col}_LOG'
        if col in df.columns:
            df = df.withColumn(new_col_name, F.log1p(F.col(col)))
    return df

In [0]:
def imputar_nulos_sampling_aleatorio(df: DataFrame) -> DataFrame:
    for col in COLUMNAS_NULOS_SAMPLING_ALEATORIO:
        if col in df.columns:
            stats = df.select(F.mean(col).alias("mean"), F.stddev(col).alias("std")).collect()[0]            
            mean_val = stats["mean"] if stats["mean"] is not None else 0.0
            std_val = stats["std"] if stats["std"] is not None else 0.0
            
            lower_bound = builtins.max(0.0, mean_val - std_val)
            upper_bound = mean_val + std_val
            
            random_formula = F.lit(lower_bound) + (F.lit(upper_bound) - F.lit(lower_bound)) * F.rand(seed=42)
            df = df.withColumn(col, F.when(F.col(col).isNull(), random_formula).otherwise(F.col(col)))         
    return df

In [0]:
def imputar_nulos_mediana(df: DataFrame) -> DataFrame:
    for col in COLUMNAS_NULOS_MEDIANA:
        if col in df.columns:
            stats = df.select(F.percentile_approx(col, 0.5, 10000).alias("median")).collect()[0]
            median_val = stats["median"]
            if median_val is None:
                median_val = 0.0
            df = df.withColumn(col, F.coalesce(F.col(col), F.lit(median_val)))
            nulos_restantes = df.filter(F.col(col).isNull()).count()            
    return df

In [0]:
def imputar_nulos_categorias(df: DataFrame) -> DataFrame:
    for col in COLUMNAS_CATEGORIAS:
        if col in df.columns:
            mode_row = (df.filter(F.col(col).isNotNull())
                          .groupBy(col)
                          .count()
                          .orderBy(F.col("count").desc())
                          .first())
            mode_val = mode_row[col] if mode_row is not None else "DESCONOCIDO"
            df = df.withColumn(col, F.coalesce(F.col(col), F.lit(mode_val)))            
    return df

In [0]:
def one_hot_encoding(df: DataFrame) -> DataFrame:
    columnas_validas = [c for c in COLUMNAS_CATEGORIAS if c in df.columns]
    if not columnas_validas:
        return df
            
    num_columnas_antes = len(df.columns)
    exprs = [F.collect_set(c).alias(c) for c in columnas_validas]
    categorias_dict = df.select(*exprs).collect()[0].asDict()
    nuevas_columnas = []
    for col in columnas_validas:
        categorias = sorted(categorias_dict[col])
        for cat in categorias:
            if cat is not None:
                condicion = F.when(F.col(col) == cat, 1).otherwise(0)
                nuevas_columnas.append(condicion.alias(f"{col}_{cat}"))
    columnas_restantes = [F.col(c) for c in df.columns if c not in columnas_validas]
    df_encoded = df.select(*columnas_restantes, *nuevas_columnas)    
    return df_encoded

In [0]:
def clustering_kmeans(df_encoded: DataFrame) -> DataFrame:

    columnas_validas = [c for c in COLUMNAS_KMEANS if c in df_encoded.columns]
    if len(columnas_validas) != len(COLUMNAS_KMEANS):
        raise ValueError("Faltan columnas de COLUMNAS_KMEANS en el DataFrame.")

    assembler = VectorAssembler(
        inputCols=COLUMNAS_KMEANS, 
        outputCol="features_kmeans"
    )
    df_vector = assembler.transform(df_encoded)
    kmeans = KMeans(
        k=3, 
        initMode="k-means||",
        seed=42, 
        featuresCol="features_kmeans", 
        predictionCol="Cluster"
    )
    model = kmeans.fit(df_vector)
    df_with_cluster = model.transform(df_vector)
    df_final = df_with_cluster.drop("features_kmeans")
    num_filas = df_final.count()
    num_columnas = len(df_final.columns)    
    return df_final

In [0]:
def procesed_data(df_encoded: DataFrame) -> DataFrame:
    existing_cols_to_drop = [col for col in COLUMNAS_DROP if col in df_encoded.columns]
    df_processed = df_encoded.drop(*existing_cols_to_drop)
    num_filas = df_processed.count()
    num_columnas = len(df_processed.columns)    
    return df_processed

In [0]:
df_columnas_negativas = imputar_columnas_negativas(df_renovacion)

In [0]:
df_transformar_columnas_log= transformar_columnas(df_columnas_negativas)

In [0]:
df_nulos_sampling_aleatorio = imputar_nulos_sampling_aleatorio(df_transformar_columnas_log)

In [0]:
df_nulos_sampling_mediana = imputar_nulos_mediana(df_nulos_sampling_aleatorio)

In [0]:
df_nulos_sampling_categoria = imputar_nulos_categorias(df_nulos_sampling_mediana)

In [0]:
df_encoding = one_hot_encoding(df_nulos_sampling_categoria)

In [0]:
df_clustering = clustering_kmeans(df_encoding)

In [0]:
df_processed = procesed_data(df_clustering)

In [0]:
df_processed.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.renovacion_prestamo_transformed")